# Step 5 Part G: Feedforward architecture ablation

The original proposal specified comparing the LSTM against a feedforward network with a fixed lookback window, to test whether recurrence (unlimited memory of the whole episode) is actually earning its complexity, or whether a much simpler memoryless-beyond-the-window model does just as well. Same loss (CVaR + turnover penalty), same training setup as the LSTM v2, so the ONLY thing that changes is architecture.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import json
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BTC_TRANSACTION_COST_RATE = 0.0005
LOOKBACK_WINDOW = 4  # feedforward network sees only the last 4 steps of features, no memory beyond that

train_data = np.load("train_episode_tensors.npz")
train_features = train_data["features"]   # (n, 24, 5)
train_spots = torch.tensor(train_data["spots"], dtype=torch.float32)
train_option_pnls = torch.tensor(train_data["option_pnls"], dtype=torch.float32)
train_masks_np = train_data["masks"]
train_masks = torch.tensor(train_masks_np, dtype=torch.float32)

val_data = np.load("val_episode_tensors.npz")
val_features = val_data["features"]
val_spots = torch.tensor(val_data["spots"], dtype=torch.float32).to(device)
val_option_pnls = torch.tensor(val_data["option_pnls"], dtype=torch.float32).to(device)
val_masks_np = val_data["masks"]
val_masks = torch.tensor(val_masks_np, dtype=torch.float32).to(device)

def build_windowed(features, window=LOOKBACK_WINDOW):
    """features: (n_episodes, seq_len, n_feat) -> (n_episodes, seq_len, window*n_feat)
    At step t, concatenate features from max(0,t-window+1)..t, zero-padding at the start of the episode."""
    n_ep, seq_len, n_feat = features.shape
    padded = np.concatenate([np.zeros((n_ep, window - 1, n_feat)), features], axis=1)  # pad at the start
    windowed = np.zeros((n_ep, seq_len, window * n_feat))
    for t in range(seq_len):
        chunk = padded[:, t:t + window, :]  # (n_ep, window, n_feat)
        windowed[:, t, :] = chunk.reshape(n_ep, -1)
    return windowed

train_windowed = torch.tensor(build_windowed(train_features), dtype=torch.float32)
val_windowed = torch.tensor(build_windowed(val_features), dtype=torch.float32).to(device)
print(f"Train windowed shape: {train_windowed.shape}, Val windowed shape: {val_windowed.shape}")

## Feedforward policy (applies the SAME small MLP independently at every timestep -- no hidden state carried between steps)

In [ ]:
class FeedforwardHedgePolicy(nn.Module):
    def __init__(self, input_size, hidden_size=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, 1),
        )
    def forward(self, x):
        # x: (batch, seq_len, window*n_feat) -- apply the MLP independently at each timestep
        raw = self.net(x)  # (batch, seq_len, 1)
        return 1.5 * torch.tanh(raw.squeeze(-1))

def simulate_pnl_batch(positions, spots, option_pnls, masks, cost_rate=BTC_TRANSACTION_COST_RATE):
    batch_size, seq_len = positions.shape
    prev_position = torch.cat([torch.zeros(batch_size, 1, device=positions.device), positions[:, :-1]], dim=1)
    trade = (positions - prev_position) * masks
    cost = trade.abs() * spots * (cost_rate / 2)
    prev_spot = torch.cat([spots[:, :1], spots[:, :-1]], dim=1)
    hedge_pnl = prev_position * (spots - prev_spot)
    total_pnl_per_step = (option_pnls + hedge_pnl - cost) * masks
    turnover = trade.abs().sum(dim=1)
    return total_pnl_per_step.sum(dim=1), turnover

def cvar_loss(terminal_pnl, alpha=0.95):
    losses = -terminal_pnl
    k = max(1, int((1 - alpha) * losses.shape[0]))
    worst_losses, _ = torch.topk(losses, k)
    return worst_losses.mean()

TURNOVER_PENALTY_WEIGHT = 100.0  # increased from v2's 5.0, which we found was too small to matter

def combined_loss(terminal_pnl, turnover, alpha=0.95, penalty_weight=TURNOVER_PENALTY_WEIGHT):
    return cvar_loss(terminal_pnl, alpha) + penalty_weight * turnover.mean()

## Train (same setup as LSTM v2: early stopping, weight decay)

In [ ]:
input_size = LOOKBACK_WINDOW * 5
model_ff = FeedforwardHedgePolicy(input_size=input_size).to(device)
optimizer = torch.optim.Adam(model_ff.parameters(), lr=1e-3, weight_decay=1e-5)

N_EPOCHS = 100
BATCH_SIZE = 256
PATIENCE = 10
n_train = train_windowed.shape[0]

train_losses, val_losses = [], []
best_val_loss = float("inf")
best_state = None
epochs_without_improvement = 0

for epoch in range(N_EPOCHS):
    model_ff.train()
    perm = torch.randperm(n_train)
    epoch_loss = 0.0
    n_batches = 0
    for start in range(0, n_train, BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]
        bx = train_windowed[idx].to(device)
        bs = train_spots[idx].to(device)
        bo = train_option_pnls[idx].to(device)
        bm = train_masks[idx].to(device)
        optimizer.zero_grad()
        positions = model_ff(bx)
        terminal_pnl, turnover = simulate_pnl_batch(positions, bs, bo, bm)
        loss = combined_loss(terminal_pnl, turnover)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    avg_train_loss = epoch_loss / n_batches
    train_losses.append(avg_train_loss)

    model_ff.eval()
    with torch.no_grad():
        vp = model_ff(val_windowed)
        vt, vturn = simulate_pnl_batch(vp, val_spots, val_option_pnls, val_masks)
        val_loss = combined_loss(vt, vturn).item()
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model_ff.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch % 5 == 0 or epoch == N_EPOCHS - 1:
        print(f"Epoch {epoch:3d} | train loss: {avg_train_loss:9.4f} | val loss: {val_loss:9.4f} | val turnover: {vturn.mean().item():.4f}")

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

print(f"\nBest val loss: {best_val_loss:.4f}")
torch.save(best_state, "best_feedforward_model.pt")
print("Saved best_feedforward_model.pt")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, label="Train")
ax.plot(val_losses, label="Val")
ax.legend()
ax.set_title("Feedforward policy training curve")
plt.tight_layout()
plt.show()